# Interseção SICOR Cerrado × Unidades de Conservação por ID_GLEBA2

Este notebook cruza o arquivo `sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv` com o shapefile `ucs.shp`.

## Regra principal

- cada `ID_GLEBA2` distinto é submetido à interseção espacial somente uma vez;
- quando o mesmo `ID_GLEBA2` aparece em várias linhas, o resultado calculado é reaproveitado;
- antes do processamento, o notebook valida se um mesmo `ID_GLEBA2` está sempre associado à mesma geometria;
- a deduplicação técnica usa a chave `ID_GLEBA2 + UC_ID_INTERNO`, e não `REF_BACEN`.

## Saídas

1. **Resultado espacial único:** uma linha por `ID_GLEBA2 × Unidade de Conservação`.
2. **Resultado detalhado expandido:** associa a resposta espacial a todas as linhas originais que possuem aquele `ID_GLEBA2`.
3. **Resumo por ID_GLEBA2:** contém todos os IDs, inclusive os que não intersectaram Unidades de Conservação.

Os atributos do shapefile são preservados com o prefixo `UC_`. Assim, uma geometria repetida não é testada novamente, mas nenhum vínculo com `REF_BACEN`, ordem, índice ou ano é perdido.


In [1]:
# ============================================================
# 1. INSTALAR, IMPORTAR E MONTAR O GOOGLE DRIVE
# ============================================================

%pip install -q --upgrade geopandas pyogrio shapely

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import force_2d, intersection as shapely_intersection, make_valid, set_precision
from shapely.errors import GEOSException

warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 180)

print('Ambiente preparado.')


Mounted at /content/drive
Ambiente preparado.


In [2]:
# ============================================================
# 2. CAMINHOS E PARÂMETROS
# ============================================================

ARQUIVO_SICOR = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/"
    "sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv"
)

ARQUIVO_UC = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Base/ucs.shp"
)

# Pasta nova para não sobrescrever resultados anteriores.
PASTA_SAIDA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/intersecoes/unidades de conservacao/"
    "resultados_por_id_gleba2"
)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_INTERSECAO_POR_ID = (
    PASTA_SAIDA / "intersecao_ucs_unica_por_id_gleba2.csv"
)

ARQUIVO_INTERSECAO_EXPANDIDA = (
    PASTA_SAIDA / "intersecao_sicor_ucs_expandida_por_id_gleba2.csv"
)

ARQUIVO_RESUMO_POR_ID = (
    PASTA_SAIDA / "resumo_ucs_por_id_gleba2.csv"
)

CRS_SICOR = "EPSG:4326"
CRS_AREA = "EPSG:5880"
TAMANHO_BLOCO = 20_000
SEPARADOR_SAIDA = ";"

for caminho in [ARQUIVO_SICOR, ARQUIVO_UC]:
    print(f"{'OK' if caminho.exists() else 'NÃO ENCONTRADO'} | {caminho}")

if not ARQUIVO_SICOR.exists():
    raise FileNotFoundError(f"CSV do SICOR não encontrado: {ARQUIVO_SICOR}")

if not ARQUIVO_UC.exists():
    raise FileNotFoundError(
        f"Shapefile de Unidades de Conservação não encontrado: "
        f"{ARQUIVO_UC}"
    )

for extensao in [".dbf", ".shx", ".prj"]:
    componente = ARQUIVO_UC.with_suffix(extensao)
    print(f"{'OK' if componente.exists() else 'ATENÇÃO'} | {componente.name}")

print("\nPasta de saída:")
print(PASTA_SAIDA)


OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/ucs.shp
OK | ucs.dbf
OK | ucs.shx
OK | ucs.prj

Pasta de saída:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/unidades de conservacao/resultados_por_id_gleba2


In [3]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

CODIFICACOES = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
SEPARADORES = [';', ',', '\t', '|']


def remover_acentos(texto):
    return ''.join(
        caractere
        for caractere in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(caractere)
    )


def normalizar_nome_coluna(nome):
    nome = remover_acentos(nome).upper().strip().replace('#', '')
    nome = re.sub(r'[^A-Z0-9]+', '_', nome)
    return re.sub(r'_+', '_', nome).strip('_')


def normalizar_colunas(df):
    df = df.copy()
    df.columns = [normalizar_nome_coluna(coluna) for coluna in df.columns]

    if df.columns.duplicated().any():
        duplicadas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f'A normalização produziu colunas duplicadas: {duplicadas}')

    return df


def detectar_csv(caminho):
    ultimo_erro = None

    for encoding in CODIFICACOES:
        for separador in SEPARADORES:
            try:
                teste = pd.read_csv(
                    caminho,
                    sep=separador,
                    encoding=encoding,
                    nrows=5,
                    dtype=str,
                    low_memory=False,
                )

                if teste.shape[1] <= 1:
                    continue

                teste = normalizar_colunas(teste)
                return encoding, separador, list(teste.columns)

            except Exception as erro:
                ultimo_erro = erro

    raise RuntimeError(
        'Não foi possível detectar a estrutura do CSV. '
        f'Último erro: {ultimo_erro}'
    )


def localizar_coluna(colunas, candidatos, obrigatoria=True):
    for candidato in [normalizar_nome_coluna(c) for c in candidatos]:
        if candidato in colunas:
            return candidato

    if obrigatoria:
        raise KeyError(
            'Nenhuma das colunas esperadas foi encontrada: '
            f'{[normalizar_nome_coluna(c) for c in candidatos]}'
        )

    return None


def padronizar_codigo(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lstrip('\ufeff')
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, '<NA>': pd.NA})
    )


def corrigir_geometrias(gdf):
    quantidade_entrada = len(gdf)

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    mascara_invalida = ~gdf.geometry.is_valid
    quantidade_invalidas = int(mascara_invalida.sum())

    if quantidade_invalidas:
        try:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].make_valid()
            )
        except Exception:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].buffer(0)
            )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
        & gdf.geometry.is_valid
    ].copy()

    quantidade_descartadas = quantidade_entrada - len(gdf)
    return gdf, quantidade_invalidas, quantidade_descartadas


def serie_wkt_para_geometria(serie):
    return gpd.GeoSeries.from_wkt(
        serie.astype('string'),
        crs=CRS_SICOR,
        on_invalid='ignore',
    )


def preparar_geometria_topologica(geometria, precisao=None):
    if geometria is None:
        return None

    try:
        if geometria.is_empty:
            return None
        geometria = force_2d(geometria)

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        if precisao is not None:
            geometria = set_precision(
                geometria,
                grid_size=precisao,
                mode='valid_output',
            )

        if geometria is None or geometria.is_empty:
            return None

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        return geometria

    except Exception:
        try:
            geometria = geometria.buffer(0)
            return None if geometria.is_empty else geometria
        except Exception:
            return None


def calcular_intersecoes_seguras(geometrias_esquerda, geometrias_direita, crs):
    resultados = []
    quantidade_recuperadas = 0
    quantidade_falhas = 0

    for geometria_esquerda, geometria_direita in zip(
        geometrias_esquerda.to_numpy(),
        geometrias_direita.to_numpy(),
    ):
        esquerda = preparar_geometria_topologica(geometria_esquerda, 0.01)
        direita = preparar_geometria_topologica(geometria_direita, 0.01)

        if esquerda is None or direita is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        intersecao = None
        recuperada = False

        try:
            intersecao = shapely_intersection(esquerda, direita, grid_size=0.01)
        except (GEOSException, ValueError, TypeError):
            recuperada = True

            try:
                esquerda_10 = preparar_geometria_topologica(esquerda, 0.10)
                direita_10 = preparar_geometria_topologica(direita, 0.10)

                if esquerda_10 is not None and direita_10 is not None:
                    intersecao = shapely_intersection(
                        esquerda_10,
                        direita_10,
                        grid_size=0.10,
                    )
            except (GEOSException, ValueError, TypeError):
                intersecao = None

            if intersecao is None:
                try:
                    intersecao = esquerda.buffer(0).intersection(direita.buffer(0))
                except Exception:
                    intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        try:
            if not intersecao.is_empty and not intersecao.is_valid:
                intersecao = make_valid(intersecao)
        except Exception:
            try:
                intersecao = intersecao.buffer(0)
            except Exception:
                intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
        else:
            resultados.append(intersecao)
            if recuperada:
                quantidade_recuperadas += 1

    return (
        gpd.GeoSeries(resultados, index=pd.RangeIndex(len(resultados)), crs=crs),
        quantidade_recuperadas,
        quantidade_falhas,
    )


In [4]:
# ============================================================
# 4. LER E PREPARAR AS UNIDADES DE CONSERVAÇÃO
# ============================================================

ucs = gpd.read_file(ARQUIVO_UC, engine='pyogrio')
crs_original_uc = ucs.crs
coluna_geometria_original = ucs.geometry.name

if crs_original_uc is None:
    raise ValueError(
        'O shapefile ucs.shp não possui CRS. Verifique se o arquivo .prj '
        'está na mesma pasta do shapefile.'
    )

ucs = normalizar_colunas(ucs)
coluna_geometria_normalizada = normalizar_nome_coluna(coluna_geometria_original)

colunas_geometricas = [
    coluna
    for coluna in ucs.columns
    if isinstance(ucs[coluna].dtype, gpd.array.GeometryDtype)
]

if coluna_geometria_normalizada not in ucs.columns:
    if not colunas_geometricas:
        raise KeyError('Nenhuma coluna geométrica foi localizada no shapefile de Unidades de Conservação.')
    coluna_geometria_normalizada = colunas_geometricas[0]

ucs = gpd.GeoDataFrame(
    ucs,
    geometry=coluna_geometria_normalizada,
    crs=crs_original_uc,
)

if ucs.geometry.name != 'geometry':
    ucs = ucs.rename_geometry('geometry')

quantidade_uc_original = len(ucs)
ucs, invalidas_uc, descartadas_uc = corrigir_geometrias(ucs)
ucs = ucs.reset_index(drop=True)
ucs['UC_ID_INTERNO'] = np.arange(1, len(ucs) + 1)

renomear_uc = {
    coluna: f'UC_{coluna}'
    for coluna in ucs.columns
    if coluna not in {'geometry', 'UC_ID_INTERNO'}
}
ucs = ucs.rename(columns=renomear_uc).set_geometry('geometry')
ucs = ucs.to_crs(CRS_SICOR).reset_index(drop=True)

ucs_area = ucs.to_crs(CRS_AREA)
ucs['UC_AREA_TOTAL_HA'] = ucs_area.geometry.area / 10_000

COLUNAS_UC_SAIDA = [coluna for coluna in ucs.columns if coluna != 'geometry']

print('Feições originais de Unidades de Conservação:', f'{quantidade_uc_original:,}')
print('Geometrias inválidas corrigidas:', f'{invalidas_uc:,}')
print('Geometrias descartadas:', f'{descartadas_uc:,}')
print('Feições válidas finais:', f'{len(ucs):,}')
print('CRS espacial:', ucs.crs)
print('CRS para áreas:', CRS_AREA)

display(ucs.head())


Feições originais de Unidades de Conservação: 646
Geometrias inválidas corrigidas: 5
Geometrias descartadas: 0
Feições válidas finais: 646
CRS espacial: EPSG:4326
CRS para áreas: EPSG:5880


,UC_GML_ID,UC_DOCLEG_ID,UC_UC_ID,UC_CD_CNUC,UC_WDPA_PID,UC_NOME_UC,UC_CRIA_ANO,UC_CRIA_ATO,UC_OUTRO_ATO,UC_PL_MANEJO,UC_CO_GESTOR,UC_QUALI_POL,UC_PPGR,UC_HA_TOTAL,UC_HA_ATO,UC_ESFERA,UC_UF,UC_MUNICIPIO,UC_ORG_GESTOR,UC_GRUPO,UC_CATEGORIA,UC_CAT_IUCN,UC_AMAZONIA,UC_CAATINGA,UC_CERRADO,UC_MATLANTICA,UC_PAMPA,UC_PANTANAL,UC_MARINHO,UC_SITUACAO,UC_LIMITE,geometry,UC_ID_INTERNO,UC_AREA_TOTAL_HA
0,None,1602,1487,0000.17.1487,352013,PARQUE ESTADUAL DO CANTÃO,08-04-2002,Lei ordinária 996 de 14-07-1998,Lei ordinária 1.319 de 04-04-2002 Ampliação,Sim,Sim,Polígono corresponde ao memorial descritivo,Programa ARPA - Áreas Protegidas da Amazônia,100412.26,88928.88,Estadual,TOCANTINS,PIUM (TO),INSTITUTO NATUREZA DO TOCANTINS - TO,Proteção Integral,Parque,Category II,None,None,301236.78,None,None,None,None,Ativo,uc,"POLYGON ((-50.12312 -9.90106, -50.12307 -9.90022, -50.12251 -9.89903, -50.12211 -9.89795, -50.12172 -9.89716, -50.12098 -9.89648, -50.12024 -9.89545, -50.1195 -9.89466, -50.119...",1,100645.169462
1,None,1624,1486,0000.17.1486,352018,PARQUE ESTADUAL DO JALAPÃO,12-01-2001,Lei ordinária 1.203 de 12-01-2001,Sem informação,Sim,Sim,Polígono corresponde ao memorial descritivo,Mosaico Jalapão,158969.84,158885,Estadual,TOCANTINS,MATEIROS (TO),INSTITUTO NATUREZA DO TOCANTINS - TO,Proteção Integral,Parque,Category II,None,None,158969.84,None,None,None,None,Ativo,uc,"POLYGON ((-46.60822 -10.22542, -46.608 -10.22588, -46.60783 -10.22623, -46.60793 -10.22659, -46.60827 -10.22675, -46.60872 -10.22703, -46.60867 -10.23102, -46.60873 -10.2314, -...",2,160232.609492
2,None,1625,1488,0000.17.1488,352020,PARQUE ESTADUAL DO LAJEADO,11-05-2001,Lei ordinária 1.224 de 11-05-2001,Sem informação,Sim,Sem informação,Polígono corresponde ao memorial descritivo,None,10750.15,9930.92,Estadual,TOCANTINS,PALMAS (TO),INSTITUTO NATUREZA DO TOCANTINS - TO,Proteção Integral,Parque,Category II,None,None,10750.15,None,None,None,None,Ativo,uc,"POLYGON ((-48.25772 -10.00699, -48.2573 -10.00723, -48.25713 -10.00735, -48.25651 -10.00776, -48.25613 -10.00817, -48.25581 -10.00856, -48.25578 -10.00863, -48.25537 -10.009, -...",3,10803.394141
3,None,1712,1491,0000.17.1491,478483,ÁREA DE PROTEÇÃO AMBIENTAL DAS NASCENTES DE ARAGUAINA,09-12-1999,Lei ordinária 1.116 de 09-12-1999,Sem informação,Sem informação,Sim,Polígono corresponde ao memorial descritivo,None,15596.92,15821.5,Estadual,TOCANTINS,"ARAGUAÍNA (TO), BABAÇULÂNDIA (TO), WANDERLÂNDIA (TO)",INSTITUTO NATUREZA DO TOCANTINS - TO,Uso Sustentável,Área de Proteção Ambiental,Category V,None,None,15596.92,None,None,None,None,Ativo,uc,"POLYGON ((-48.10623 -7.30802, -48.17532 -7.22607, -48.1583 -7.1897, -48.18816 -7.17193, -48.18339 -7.12694, -48.13663 -7.12202, -48.08552 -7.16454, -48.08081 -7.2464, -48.09281...",4,15678.371852
4,None,1713,1494,0000.17.1494,351903,ÁREA DE PROTEÇÃO AMBIENTAL LAGO DE PEIXE/ANGICAL,20-03-2002,Decreto 1.444 de 18-03-2002,Decreto 1.444 de 18-03-2002 Retificação,Sem informação,Sem informação,Polígono corresponde ao memorial descritivo,None,75450.75,78873.82,Estadual,TOCANTINS,"PARANÃ (TO), PEIXE (TO), SÃO SALVADOR DO TOCANTINS (TO)",INSTITUTO NATUREZA DO TOCANTINS - TO,Uso Sustentável,Área de Proteção Ambiental,Category V,None,None,150901.5,None,None,None,None,Ativo,uc,"POLYGON ((-47.85151 -12.65017, -47.85185 -12.65033, -47.85227 -12.65046, -47.85742 -12.65193, -47.85794 -12.65205, -47.85889 -12.65215, -47.85985 -12.65208, -47.86079 -12.65184...",5,75821.740192


In [5]:
# ============================================================
# 5. DETECTAR E VALIDAR AS COLUNAS DO SICOR
# ============================================================

encoding_sicor, separador_sicor, colunas_sicor = detectar_csv(ARQUIVO_SICOR)

COLUNA_ID_GLEBA2 = localizar_coluna(
    colunas_sicor,
    ['ID_GLEBA2', 'ID_GLEBA_2'],
)

COLUNA_GEOMETRIA_SICOR = localizar_coluna(
    colunas_sicor,
    ['GT_GEOMETRIA', 'GT_GEOMETRIA_REPRESENTATIVA', 'GEOMETRIA', 'GEOMETRY', 'WKT'],
)

COLUNA_REF_BACEN = localizar_coluna(
    colunas_sicor,
    ['REF_BACEN', '#REF_BACEN', 'CD_REF_BACEN'],
    obrigatoria=False,
)

print('Codificação:', encoding_sicor)
print('Separador:', repr(separador_sicor))
print('Coluna-chave:', COLUNA_ID_GLEBA2)
print('Coluna de geometria:', COLUNA_GEOMETRIA_SICOR)
print('Coluna REF_BACEN:', COLUNA_REF_BACEN)
print('Quantidade de colunas:', len(colunas_sicor))

if COLUNA_ID_GLEBA2 == COLUNA_GEOMETRIA_SICOR:
    raise RuntimeError('A coluna do ID e a coluna geométrica não podem ser a mesma.')


Codificação: utf-8-sig
Separador: ';'
Coluna-chave: ID_GLEBA2
Coluna de geometria: GT_GEOMETRIA
Coluna REF_BACEN: REF_BACEN
Quantidade de colunas: 27


In [6]:
# ============================================================
# 6. INTERSECTAR UMA ÚNICA VEZ CADA ID_GLEBA2
# ============================================================

# Guarda somente uma assinatura numérica da geometria de cada ID.
# Isso permite reconhecer IDs já processados sem manter todos os WKT na memória.
assinatura_geometria_por_id = {}
resultados_intersecao = []

total_linhas_lidas = 0
total_linhas_com_id_e_geometria = 0
total_ids_novos = 0
total_repeticoes_puladas = 0
total_ids_geometria_valida = 0
total_geometrias_invalidas_corrigidas = 0
total_geometrias_descartadas = 0
total_candidatos_espaciais = 0
total_intersecoes_validas = 0
total_intersecoes_recuperadas = 0
total_falhas_topologicas = 0
total_contatos_sem_area = 0

leitor = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    bloco = normalizar_colunas(bloco)
    total_linhas_lidas += len(bloco)

    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    bloco[COLUNA_GEOMETRIA_SICOR] = bloco[COLUNA_GEOMETRIA_SICOR].astype('string').str.strip()

    bloco = bloco.dropna(
        subset=[COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
    ).copy()

    total_linhas_com_id_e_geometria += len(bloco)

    if bloco.empty:
        print(f'Bloco {numero_bloco:03d}: sem ID_GLEBA2 e geometria válidos')
        continue

    bloco['_HASH_GEOMETRIA'] = (
        pd.util.hash_pandas_object(
            bloco[COLUNA_GEOMETRIA_SICOR],
            index=False,
        )
        .astype('uint64')
    )

    # Validação dentro do próprio bloco.
    geometrias_por_id_no_bloco = bloco.groupby(
        COLUNA_ID_GLEBA2,
        dropna=False,
    )['_HASH_GEOMETRIA'].nunique()

    ids_inconsistentes = geometrias_por_id_no_bloco[
        geometrias_por_id_no_bloco > 1
    ].index.tolist()

    if ids_inconsistentes:
        raise ValueError(
            'Um mesmo ID_GLEBA2 foi associado a geometrias diferentes. '
            f'Primeiros IDs inconsistentes: {ids_inconsistentes[:10]}'
        )

    representantes = bloco.drop_duplicates(
        subset=[COLUNA_ID_GLEBA2],
        keep='first',
    ).reset_index(drop=True)

    mascara_id_novo = []

    # Validação também contra IDs que apareceram em blocos anteriores.
    for id_atual, hash_atual in representantes[
        [COLUNA_ID_GLEBA2, '_HASH_GEOMETRIA']
    ].itertuples(index=False, name=None):
        hash_atual = int(hash_atual)

        if id_atual in assinatura_geometria_por_id:
            if assinatura_geometria_por_id[id_atual] != hash_atual:
                raise ValueError(
                    f'O ID_GLEBA2 {id_atual} apareceu com geometrias diferentes '
                    'em blocos distintos do CSV.'
                )
            mascara_id_novo.append(False)
        else:
            assinatura_geometria_por_id[id_atual] = hash_atual
            mascara_id_novo.append(True)

    representantes_novos = representantes.loc[
        mascara_id_novo
    ].copy().reset_index(drop=True)

    total_ids_novos += len(representantes_novos)
    total_repeticoes_puladas += len(bloco) - len(representantes_novos)

    if representantes_novos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            'todos os IDs já haviam sido processados'
        )
        del bloco, representantes, representantes_novos
        gc.collect()
        continue

    geometria = serie_wkt_para_geometria(
        representantes_novos[COLUNA_GEOMETRIA_SICOR]
    )

    gdf_ids = gpd.GeoDataFrame(
        representantes_novos[
            [COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
        ].copy(),
        geometry=geometria,
        crs=CRS_SICOR,
    )

    gdf_ids, invalidas_corrigidas, descartadas = corrigir_geometrias(gdf_ids)
    gdf_ids = gdf_ids.reset_index(drop=True)

    total_geometrias_invalidas_corrigidas += invalidas_corrigidas
    total_geometrias_descartadas += descartadas
    total_ids_geometria_valida += len(gdf_ids)

    if gdf_ids.empty:
        print(
            f'Bloco {numero_bloco:03d}: IDs novos={len(representantes_novos):,} | '
            'nenhuma geometria válida'
        )
        del bloco, representantes, representantes_novos, gdf_ids
        gc.collect()
        continue

    candidatos = gpd.sjoin(
        gdf_ids,
        ucs,
        how='inner',
        predicate='intersects',
    ).reset_index(drop=True)

    total_candidatos_espaciais += len(candidatos)

    if candidatos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            f'IDs novos={len(representantes_novos):,} | interseções=0 | '
            f'repetições puladas={len(bloco) - len(representantes_novos):,}'
        )
        del bloco, representantes, representantes_novos, gdf_ids, candidatos
        gc.collect()
        continue

    candidatos['GEOMETRIA_UC'] = (
        candidatos['index_right'].map(ucs.geometry)
    )

    candidatos = candidatos.dropna(
        subset=['GEOMETRIA_UC']
    ).reset_index(drop=True)

    geometria_gleba_wgs84 = gpd.GeoSeries(
        candidatos.geometry.to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_uc_wgs84 = gpd.GeoSeries(
        candidatos['GEOMETRIA_UC'].to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_gleba_area = geometria_gleba_wgs84.to_crs(CRS_AREA)
    geometria_uc_area = geometria_uc_wgs84.to_crs(CRS_AREA)

    geometria_gleba_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_gleba_area],
        index=pd.RangeIndex(len(geometria_gleba_area)),
        crs=CRS_AREA,
    )

    geometria_uc_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_uc_area],
        index=pd.RangeIndex(len(geometria_uc_area)),
        crs=CRS_AREA,
    )

    (
        geometria_intersecao_area,
        recuperadas_bloco,
        falhas_bloco,
    ) = calcular_intersecoes_seguras(
        geometria_gleba_area,
        geometria_uc_area,
        CRS_AREA,
    )

    total_intersecoes_recuperadas += recuperadas_bloco
    total_falhas_topologicas += falhas_bloco

    candidatos['AREA_GLEBA_HA_CALCULADA'] = (
        geometria_gleba_area.area.to_numpy() / 10_000
    )
    candidatos['AREA_INTERSECAO_HA'] = (
        geometria_intersecao_area.area.to_numpy() / 10_000
    )
    candidatos['PERCENTUAL_GLEBA_INTERSECTADA'] = np.where(
        candidatos['AREA_GLEBA_HA_CALCULADA'] > 0,
        candidatos['AREA_INTERSECAO_HA']
        / candidatos['AREA_GLEBA_HA_CALCULADA']
        * 100,
        np.nan,
    )

    mascara_area_positiva = (
        candidatos['AREA_INTERSECAO_HA'].fillna(0).gt(0).to_numpy()
    )
    total_contatos_sem_area += int((~mascara_area_positiva).sum())

    if not mascara_area_positiva.any():
        print(
            f'Bloco {numero_bloco:03d}: candidatos={len(candidatos):,} | '
            'nenhuma interseção com área positiva'
        )
        del (
            bloco, representantes, representantes_novos, gdf_ids, candidatos,
            geometria_gleba_wgs84, geometria_uc_wgs84,
            geometria_gleba_area, geometria_uc_area,
            geometria_intersecao_area, mascara_area_positiva,
        )
        gc.collect()
        continue

    candidatos = candidatos.loc[mascara_area_positiva].reset_index(drop=True)

    geometria_intersecao_area_valida = gpd.GeoSeries(
        geometria_intersecao_area.to_numpy()[mascara_area_positiva],
        index=pd.RangeIndex(int(mascara_area_positiva.sum())),
        crs=CRS_AREA,
    )
    geometria_intersecao_wgs84 = geometria_intersecao_area_valida.to_crs(CRS_SICOR)

    candidatos['GEOMETRIA_UC_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in candidatos['GEOMETRIA_UC']
    ]
    candidatos['GEOMETRIA_INTERSECAO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in geometria_intersecao_wgs84
    ]
    candidatos['TIPO_IMPEDIMENTO'] = 'Unidade de Conservação'

    colunas_resultado = [
        COLUNA_ID_GLEBA2,
        *COLUNAS_UC_SAIDA,
        'AREA_GLEBA_HA_CALCULADA',
        'AREA_INTERSECAO_HA',
        'PERCENTUAL_GLEBA_INTERSECTADA',
        'GEOMETRIA_UC_WKT',
        'GEOMETRIA_INTERSECAO_WKT',
        'TIPO_IMPEDIMENTO',
    ]

    resultado_bloco = pd.DataFrame(candidatos[colunas_resultado])
    resultados_intersecao.append(resultado_bloco)
    total_intersecoes_validas += len(resultado_bloco)

    print(
        f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
        f'IDs novos={len(representantes_novos):,} | '
        f'repetições puladas={len(bloco) - len(representantes_novos):,} | '
        f'candidatos={len(mascara_area_positiva):,} | '
        f'interseções com área={len(resultado_bloco):,}'
    )

    del (
        bloco, representantes, representantes_novos, gdf_ids, candidatos,
        geometria_gleba_wgs84, geometria_uc_wgs84,
        geometria_gleba_area, geometria_uc_area,
        geometria_intersecao_area, geometria_intersecao_area_valida,
        geometria_intersecao_wgs84, mascara_area_positiva, resultado_bloco,
    )
    gc.collect()

print('\nPROCESSAMENTO ESPACIAL CONCLUÍDO')
print('=' * 90)
print('Linhas lidas:', f'{total_linhas_lidas:,}')
print('Linhas com ID e geometria:', f'{total_linhas_com_id_e_geometria:,}')
print('ID_GLEBA2 distintos:', f'{len(assinatura_geometria_por_id):,}')
print('IDs submetidos ao teste:', f'{total_ids_novos:,}')
print('Repetições que não foram testadas novamente:', f'{total_repeticoes_puladas:,}')
print('IDs com geometria válida:', f'{total_ids_geometria_valida:,}')
print('Geometrias inválidas corrigidas:', f'{total_geometrias_invalidas_corrigidas:,}')
print('Geometrias descartadas:', f'{total_geometrias_descartadas:,}')
print('Candidatos espaciais:', f'{total_candidatos_espaciais:,}')
print('Interseções recuperadas:', f'{total_intersecoes_recuperadas:,}')
print('Falhas topológicas:', f'{total_falhas_topologicas:,}')
print('Contatos sem área positiva:', f'{total_contatos_sem_area:,}')
print('Interseções válidas ID_GLEBA2 × Unidade de Conservação:', f'{total_intersecoes_validas:,}')


Bloco 001: lidas=20,000 | IDs novos=18,663 | repetições puladas=1,337 | candidatos=765 | interseções com área=763
Bloco 002: lidas=20,000 | IDs novos=17,690 | repetições puladas=2,310 | candidatos=899 | interseções com área=899
Bloco 003: lidas=20,000 | IDs novos=17,256 | repetições puladas=2,744 | candidatos=687 | interseções com área=687
Bloco 004: lidas=20,000 | IDs novos=17,061 | repetições puladas=2,939 | candidatos=607 | interseções com área=607
Bloco 005: lidas=20,000 | IDs novos=16,706 | repetições puladas=3,294 | candidatos=551 | interseções com área=551
Bloco 006: lidas=20,000 | IDs novos=14,364 | repetições puladas=5,636 | candidatos=574 | interseções com área=574
Bloco 007: lidas=20,000 | IDs novos=15,135 | repetições puladas=4,865 | candidatos=806 | interseções com área=806
Bloco 008: lidas=20,000 | IDs novos=15,230 | repetições puladas=4,770 | candidatos=614 | interseções com área=614
Bloco 009: lidas=20,000 | IDs novos=14,954 | repetições puladas=5,046 | candidatos=504 |

In [7]:
# ============================================================
# 7. CONSOLIDAR E SALVAR O RESULTADO ÚNICO POR ID_GLEBA2
# ============================================================

if resultados_intersecao:
    intersecoes_por_id = pd.concat(
        resultados_intersecao,
        ignore_index=True,
        sort=False,
    )
else:
    intersecoes_por_id = pd.DataFrame(
        columns=[COLUNA_ID_GLEBA2, 'UC_ID_INTERNO']
    )

del resultados_intersecao
gc.collect()

quantidade_antes = len(intersecoes_por_id)

if not intersecoes_por_id.empty:
    intersecoes_por_id = (
        intersecoes_por_id
        .sort_values('AREA_INTERSECAO_HA', ascending=False)
        .drop_duplicates(
            subset=[COLUNA_ID_GLEBA2, 'UC_ID_INTERNO'],
            keep='first',
        )
        .reset_index(drop=True)
    )

print(
    'Duplicações técnicas ID_GLEBA2 × Unidade de Conservação removidas:',
    f'{quantidade_antes - len(intersecoes_por_id):,}',
)

intersecoes_por_id.to_csv(
    ARQUIVO_INTERSECAO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

# O resumo inclui todos os IDs, inclusive os sem interseção.
todos_ids = pd.DataFrame({
    COLUNA_ID_GLEBA2: list(assinatura_geometria_por_id.keys())
})

if intersecoes_por_id.empty:
    resumo_por_id = todos_ids.copy()
    resumo_por_id['TEM_INTERSECAO_UC'] = False
    resumo_por_id['QUANTIDADE_UNIDADES_CONSERVACAO_INTERSECTADAS'] = 0
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = 0.0
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = 0.0
else:
    resumo_positivo = (
        intersecoes_por_id.groupby(COLUNA_ID_GLEBA2, dropna=False)
        .agg(
            QUANTIDADE_UNIDADES_CONSERVACAO_INTERSECTADAS=(
                'UC_ID_INTERNO', 'nunique'
            ),
            AREA_INTERSECAO_TOTAL_HA=('AREA_INTERSECAO_HA', 'sum'),
            PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA=(
                'PERCENTUAL_GLEBA_INTERSECTADA', 'max'
            ),
        )
        .reset_index()
    )

    resumo_por_id = todos_ids.merge(
        resumo_positivo,
        on=COLUNA_ID_GLEBA2,
        how='left',
        validate='one_to_one',
    )

    resumo_por_id['TEM_INTERSECAO_UC'] = (
        resumo_por_id['QUANTIDADE_UNIDADES_CONSERVACAO_INTERSECTADAS']
        .notna()
    )
    resumo_por_id['QUANTIDADE_UNIDADES_CONSERVACAO_INTERSECTADAS'] = (
        resumo_por_id['QUANTIDADE_UNIDADES_CONSERVACAO_INTERSECTADAS']
        .fillna(0)
        .astype('int64')
    )
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = (
        resumo_por_id['AREA_INTERSECAO_TOTAL_HA'].fillna(0.0)
    )
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = (
        resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'].fillna(0.0)
    )

    resumo_por_id = resumo_por_id[
        [
            COLUNA_ID_GLEBA2,
            'TEM_INTERSECAO_UC',
            'QUANTIDADE_UNIDADES_CONSERVACAO_INTERSECTADAS',
            'AREA_INTERSECAO_TOTAL_HA',
            'PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA',
        ]
    ]

resumo_por_id.to_csv(
    ARQUIVO_RESUMO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

print('\nResultado único salvo em:')
print(ARQUIVO_INTERSECAO_POR_ID)
print('\nResumo de todos os IDs salvo em:')
print(ARQUIVO_RESUMO_POR_ID)
print('\nIDs com interseção:', f"{resumo_por_id['TEM_INTERSECAO_UC'].sum():,}")
print('IDs sem interseção:', f"{(~resumo_por_id['TEM_INTERSECAO_UC']).sum():,}")


Duplicações técnicas ID_GLEBA2 × Unidade de Conservação removidas: 0

Resultado único salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/unidades de conservacao/resultados_por_id_gleba2/intersecao_ucs_unica_por_id_gleba2.csv

Resumo de todos os IDs salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/unidades de conservacao/resultados_por_id_gleba2/resumo_ucs_por_id_gleba2.csv

IDs com interseção: 14,621
IDs sem interseção: 380,954


In [8]:
# ============================================================
# 8. DEVOLVER A RESPOSTA A TODAS AS LINHAS DO SICOR
# ============================================================

# A interseção não é recalculada aqui. O merge apenas replica a resposta
# de cada ID_GLEBA2 para todos os REF_BACEN, ordens, índices e anos ligados a ele.

colunas_espaciais_para_merge = [
    coluna
    for coluna in intersecoes_por_id.columns
    if coluna != COLUNA_ID_GLEBA2
]

escrever_cabecalho = True
total_linhas_originais_reprocessadas = 0
total_linhas_expandidas = 0

leitor_expandir = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_expandir, start=1):
    bloco = normalizar_colunas(bloco)
    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    total_linhas_originais_reprocessadas += len(bloco)

    if intersecoes_por_id.empty:
        expandido = pd.DataFrame()
    else:
        expandido = bloco.merge(
            intersecoes_por_id,
            on=COLUNA_ID_GLEBA2,
            how='inner',
            validate='many_to_many',
            suffixes=('', '_INTERSECAO'),
        )

    if expandido.empty:
        print(f'Bloco {numero_bloco:03d}: nenhuma linha com interseção')
        del bloco, expandido
        gc.collect()
        continue

    expandido.to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        mode='w' if escrever_cabecalho else 'a',
        header=escrever_cabecalho,
        index=False,
        sep=SEPARADOR_SAIDA,
        decimal=',',
        encoding='utf-8-sig' if escrever_cabecalho else 'utf-8',
    )

    total_linhas_expandidas += len(expandido)
    escrever_cabecalho = False

    print(
        f'Bloco {numero_bloco:03d}: originais={len(bloco):,} | '
        f'linhas de interseção expandidas={len(expandido):,}'
    )

    del bloco, expandido
    gc.collect()

if escrever_cabecalho:
    # Nenhuma interseção: ainda cria um CSV vazio com cabeçalho conhecido.
    pd.DataFrame(
        columns=[*colunas_sicor, *colunas_espaciais_para_merge]
    ).to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        index=False,
        sep=SEPARADOR_SAIDA,
        encoding='utf-8-sig',
    )

if total_linhas_originais_reprocessadas != total_linhas_lidas:
    raise RuntimeError(
        'A quantidade de linhas relidas para a expansão difere da '
        'quantidade lida na etapa espacial.'
    )

print('\nRESULTADO DETALHADO EXPANDIDO')
print('=' * 90)
print('Linhas originais relidas:', f'{total_linhas_originais_reprocessadas:,}')
print('Linhas finais de interseção:', f'{total_linhas_expandidas:,}')
print('Arquivo salvo em:')
print(ARQUIVO_INTERSECAO_EXPANDIDA)


Bloco 001: originais=20,000 | linhas de interseção expandidas=810
Bloco 002: originais=20,000 | linhas de interseção expandidas=988
Bloco 003: originais=20,000 | linhas de interseção expandidas=792
Bloco 004: originais=20,000 | linhas de interseção expandidas=707
Bloco 005: originais=20,000 | linhas de interseção expandidas=687
Bloco 006: originais=20,000 | linhas de interseção expandidas=797
Bloco 007: originais=20,000 | linhas de interseção expandidas=1,062
Bloco 008: originais=20,000 | linhas de interseção expandidas=789
Bloco 009: originais=20,000 | linhas de interseção expandidas=639
Bloco 010: originais=20,000 | linhas de interseção expandidas=767
Bloco 011: originais=20,000 | linhas de interseção expandidas=765
Bloco 012: originais=20,000 | linhas de interseção expandidas=692
Bloco 013: originais=20,000 | linhas de interseção expandidas=823
Bloco 014: originais=20,000 | linhas de interseção expandidas=765
Bloco 015: originais=20,000 | linhas de interseção expandidas=819
Bloco 01

In [9]:
# ============================================================
# 9. CONFERÊNCIA FINAL
# ============================================================

print('ARQUIVOS GERADOS')
print('=' * 100)

for caminho in [
    ARQUIVO_INTERSECAO_POR_ID,
    ARQUIVO_INTERSECAO_EXPANDIDA,
    ARQUIVO_RESUMO_POR_ID,
]:
    print(f"{'OK' if caminho.exists() else 'NÃO CRIADO'} | {caminho}")

print('\nAMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2')
display(intersecoes_por_id.head(5))

print('\nAMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO')
display(resumo_por_id.head(5))

print('\nESTATÍSTICAS')
print('ID_GLEBA2 distintos analisados:', f'{len(resumo_por_id):,}')
print('ID_GLEBA2 com interseção:', f"{resumo_por_id['TEM_INTERSECAO_UC'].sum():,}")
print('Pares ID_GLEBA2 × Unidade de Conservação:', f'{len(intersecoes_por_id):,}')

if not intersecoes_por_id.empty:
    print(
        'Área total das interseções únicas (ha):',
        f"{intersecoes_por_id['AREA_INTERSECAO_HA'].sum():,.4f}",
    )


ARQUIVOS GERADOS
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/unidades de conservacao/resultados_por_id_gleba2/intersecao_ucs_unica_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/unidades de conservacao/resultados_por_id_gleba2/intersecao_sicor_ucs_expandida_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/unidades de conservacao/resultados_por_id_gleba2/resumo_ucs_por_id_gleba2.csv

AMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2


,ID_GLEBA2,UC_GML_ID,UC_DOCLEG_ID,UC_UC_ID,UC_CD_CNUC,UC_WDPA_PID,UC_NOME_UC,UC_CRIA_ANO,UC_CRIA_ATO,UC_OUTRO_ATO,UC_PL_MANEJO,UC_CO_GESTOR,UC_QUALI_POL,UC_PPGR,UC_HA_TOTAL,UC_HA_ATO,UC_ESFERA,UC_UF,UC_MUNICIPIO,UC_ORG_GESTOR,UC_GRUPO,UC_CATEGORIA,UC_CAT_IUCN,UC_AMAZONIA,UC_CAATINGA,UC_CERRADO,UC_MATLANTICA,UC_PAMPA,UC_PANTANAL,UC_MARINHO,UC_SITUACAO,UC_LIMITE,UC_ID_INTERNO,UC_AREA_TOTAL_HA,AREA_GLEBA_HA_CALCULADA,AREA_INTERSECAO_HA,PERCENTUAL_GLEBA_INTERSECTADA,GEOMETRIA_UC_WKT,GEOMETRIA_INTERSECAO_WKT,TIPO_IMPEDIMENTO
0,129176,None,60412,22943,0000.00.0065,2221,ESTAÇÃO ECOLÓGICA DE URUÇUÍ-UNA,04-06-1981,Decreto Nº 86.061 de 02-06-1981,Sem informação,Não,Não,Polígono corresponde ao memorial descritivo,None,129297.37,135000,Federal,PIAUÍ,BAIXA GRANDE DO RIBEIRO (PI),INSTITUTO CHICO MENDES DE CONSERVAÇÃO DA BIODIVERSIDADE,Proteção Integral,Estação Ecológica,Category Ia,None,None,129298.51,None,None,None,None,Ativo,uc,505,1.307712e+05,13956.107795,6050.489249,43.353701,"POLYGON ((-45.42902149851112 -8.895575600241951, -45.427992694999936 -8.894214670999949, -45.42765770700001 -8.892402715999955, -45.42668599799999 -8.88958957999994, -45.425492...","POLYGON ((-45.2606763677087 -9.012652518506725, -45.24982267179121 -9.018651292913308, -45.244877476516 -9.01892097632437, -45.24191381538811 -9.014957073096694, -45.2324808824...",Unidade de Conservação
1,27491,None,60412,22943,0000.00.0065,2221,ESTAÇÃO ECOLÓGICA DE URUÇUÍ-UNA,04-06-1981,Decreto Nº 86.061 de 02-06-1981,Sem informação,Não,Não,Polígono corresponde ao memorial descritivo,None,129297.37,135000,Federal,PIAUÍ,BAIXA GRANDE DO RIBEIRO (PI),INSTITUTO CHICO MENDES DE CONSERVAÇÃO DA BIODIVERSIDADE,Proteção Integral,Estação Ecológica,Category Ia,None,None,129298.51,None,None,None,None,Ativo,uc,505,1.307712e+05,13956.107795,6050.489249,43.353701,"POLYGON ((-45.42902149851112 -8.895575600241951, -45.427992694999936 -8.894214670999949, -45.42765770700001 -8.892402715999955, -45.42668599799999 -8.88958957999994, -45.425492...","POLYGON ((-45.2606763677087 -9.012652518506725, -45.24982267179121 -9.018651292913308, -45.244877476516 -9.01892097632437, -45.24191381538811 -9.014957073096694, -45.2324808824...",Unidade de Conservação
2,260078,None,18586,4094,0627.50.4094,None,ÁREA DE PROTEÇÃO AMBIENTAL DO RIO SUCURIU- PARAÍSO,14-06-2013,Lei ordinária 035/2013 de 14-06-2013,Lei ordinária 353/2021 de 18-03-2013 Reavaliação,Sim,Sim,Polígono corresponde ao memorial descritivo,None,308869.55,310538.5299,Municipal,MATO GROSSO DO SUL,"ÁGUA CLARA (MS), CAMAPUÃ (MS), CHAPADÃO DO SUL (MS), COSTA RICA (MS), PARAÍSO DAS ÁGUAS (MS)","SECRETARIA MUNICIPAL DE DESENVOLVIMENTO ECONÔMICO, MEIO AMBIENTE E TURISMO DE PARAÍSO DAS ÁGUAS - MS",Uso Sustentável,Área de Proteção Ambiental,Category V,None,None,534171.91,None,None,None,None,Ativo,uc,146,3.069220e+05,2676.348584,2640.459265,98.659019,"GEOMETRYCOLLECTION (MULTIPOLYGON (((-53.19698461767023 -18.958040488429383, -53.196905431619626 -18.958348608468825, -53.19727977636246 -18.958696209603374, -53.19631005737016 ...","MULTIPOLYGON (((-52.79045708785468 -19.460172075755903, -52.78989473142621 -19.459408910217295, -52.78862149793381 -19.458929027941995, -52.78857693636253 -19.45888907964479, -...",Unidade de Conservação
3,247069,None,20027,4330,0635.50.4330,555758892,ÁREA DE PROTEÇÃO AMBIENTAL DA BACIA DO RIO IGUATEMI NO MUNICIPIO DE PARANHOS,28-04-2003,Decreto 003 de 28 de abril de 2003 de 28-04-2003,Sem informação,Sim,Sim,Polígono é uma estimativa dos limites da UC,None,130135.58,130136.658,Municipal,MATO GROSSO DO SUL,PARANHOS (MS),PREFEITURA MUNICIPAL DE PARANHOS - MS,Uso Sustentável,Área de Proteção Ambiental,Category V,None,None,112864.64,17235.86,None,None,None,Ativo,uc,170,1.301673e+05,2474.428933,2474.428933,100.000000,"POLYGON ((-55.55258558185311 -23.483421337710148, -55.545173613036155 -23.48028565518577, -55.54111476956356 -23.480756772529148, -55.53625483754027 -23.479095471507488, -55.53...","POLYGON ((-55.3958156495441 -23.571691859790043, -


AMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO


,ID_GLEBA2,TEM_INTERSECAO_UC,QUANTIDADE_UNIDADES_CONSERVACAO_INTERSECTADAS,AREA_INTERSECAO_TOTAL_HA,PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA
0,1,False,0,0.0,0.0
1,2,False,0,0.0,0.0
2,3,False,0,0.0,0.0
3,4,False,0,0.0,0.0
4,5,False,0,0.0,0.0



ESTATÍSTICAS
ID_GLEBA2 distintos analisados: 395,575
ID_GLEBA2 com interseção: 14,621
Pares ID_GLEBA2 × Unidade de Conservação: 14,998
Área total das interseções únicas (ha): 1,287,138.0189
